# 01 — Model (Task 2 Datavidia 11.0, Tim Perintis)

**Alur:** citra → preprocessing → DINOv2 → muat cluster B + kamus nama cluster → label spreading → classifier → simpan model.

| Bagian | Sumber kode |
|---|---|
| 1. Preprocessing & ekstraksi | Hafizh (`hafizh/v3.1/dino_v4_frozen_for_seno.py`, dikunci) |
| 2. Cluster & nama kelas | Fahmi (`fahmi/B_kejadian.ipynb`, v3-7): Spectral K=40, 4 cluster campur dipecah → 46 cluster |
| 3. Label spreading & classifier | Seno (`seno/C_model_run5.ipynb`, run5_final) |

**Output:** `model/` → `model.joblib`, `pred_train.csv`, `meta.json`.

Clustering **tidak** di-fit ulang di sini: hasil B (`events.parquet`, `sub_events.parquet`) dimuat dan dicek lewat hash.
Nama kelas ditetapkan **per cluster** (bukan per foto) dari pengamatan montase oleh AI (Claude), tanpa tinjauan tim,
dan ditulis eksplisit sebagai kamus `NAMA_CLUSTER` di bagian 2. Nama tidak disetel dari skor leaderboard.
Prediksi akhir dihasilkan classifier.


## 0. Library

In [ ]:
import json, os, platform, hashlib
from datetime import datetime
from pathlib import Path
from importlib.metadata import version, PackageNotFoundError

import numpy as np
import pandas as pd
import joblib, sklearn

from sklearn.decomposition import PCA
from sklearn.semi_supervised import LabelSpreading
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline


## Pengaturan

In [ ]:
# --- Path (relatif terhadap folder notebook; sesuaikan dengan aturan pengumpulan) ---
DATA_DIR    = Path("Dataset/Fenomena")              # berisi <Id>/pre.jpg dan <Id>/post.jpg
SAMPLE_SUB  = Path("Dataset/sample_submission.csv")
WEIGHTS_DIR = None                                  # None = dicari otomatis di ./weights/dinov2_large (offline)
MODEL_DIR   = Path("model")

# --- Hasil B (Fahmi v3-7), disalin dari fahmi/v3 ---
EVENTS_PATH        = Path("events.parquet")         # Id, event_id, jarak_pusat, kedekatan, kelas_cluster, yakin_cluster
SUB_EVENTS_PATH    = Path("sub_events.parquet")     # pemetaan foto -> induk/sub (untuk cek hash)
CLUSTER_NAMES_PATH = Path("cluster_names.json")     # kelas, yakin, alasan, penilai per cluster
PCA_B_PATH         = Path("pca.joblib")             # PCA ruang Z milik B (dimuat, tidak di-fit ulang)
MEAN_X_PATH        = Path("mean_x.npy")

SEED = 42
KELAS = ["Earthquake", "Explosion", "Fire", "Hurricane", "Tsunami", "Volcano"]

# --- MODE_UJI: WAJIB False untuk versi yang dikumpulkan ---
MODE_UJI = False
FITUR_TERSIMPAN = Path("../hafizh/v3.1")            # hanya dipakai saat MODE_UJI
C_ACUAN = Path("../seno/run5_final")                # cek: prediksi di sini harus sama dengan run C
if MODE_UJI:
    SAMPLE_SUB = next(Path(p) for p in ["../../Task-2/sample_submission.csv", "../../sample_submission.csv",
                                        "../Dataset/sample_submission.csv", str(SAMPLE_SUB)] if Path(p).exists())

# --- B (sama dengan fahmi/v3/meta.json) ---
B_ROUND = 4

# --- C (sama dengan seno/run5_final/meta.json) ---
SEED_KEDEKATAN_MIN = 0.5                     # foto dengan kedekatan >= 0.5 jadi seed
BOBOT_YAKIN = {"yakin": 1.0, "ragu": 0.5}    # bobot sampel = kedekatan x BOBOT_YAKIN[yakin cluster]
LS_KNN, LS_ALPHA = 15, 0.2
FITUR_BLOK = ["color_pre", "color_post"]
C_PCA, LR_C = 128, 1.0

np.random.seed(SEED)
for p in [SAMPLE_SUB, EVENTS_PATH, SUB_EVENTS_PATH, CLUSTER_NAMES_PATH, PCA_B_PATH, MEAN_X_PATH]:
    assert p.exists(), f"{p} tidak ada"
print("MODE_UJI =", MODE_UJI, "| sample_submission:", SAMPLE_SUB)


## 1. [A — Hafizh] Preprocessing & ekstraksi fitur (DIKUNCI)

Sel berikut diisi **salinan persis** fungsi preprocessing + ekstraksi yang dikunci Hafizh (sama di `01_model` dan `02_submission`).

Kontrak keluaran `ekstrak_semua(data_dir, ids)`:

| Kunci | Isi |
|---|---|
| `emb` | dict `{"color_pre", "color_post", ...}` → `np.ndarray (N, 1024) float32`, urutan = `ids` |
| `art` | DataFrame `artifact_and_phys_v3` (kolom `Id`, urutan = `ids`) |
| `patch` | DataFrame `patch_change_stats_v3` (kolom `Id`, urutan = `ids`) |
| `phys` | DataFrame `phys_v4` (kolom `Id`, urutan = `ids`) |

Notebook ini hanya memakai `emb` (`color_pre`, `color_post`), jadi `ekstrak_semua` dipanggil dengan `compute_patch_and_phys=False` (embedding pre/post tidak berubah, hanya statistik tambahan yang dilewati).

Selama `MODE_UJI = True`, embedding dimuat dari `hafizh/v3.1` (hasil ekstraksi Kaggle yang sama) supaya sisa notebook bisa diuji cepat.

In [ ]:
# ===== Fungsi preprocessing + ekstraksi yang dikunci Hafizh =====
# Salinan persis hafizh/v3.1/dino_v4_frozen_for_seno.py (sha1 dbdcd5039f77f47064bf3d73996212adcdeb5a24).
# Jangan diubah. Sel yang sama ada di 01_model.ipynb dan 02_submission.ipynb.

"""
dino_v4_frozen_for_seno.py — Modul Terkunci A -> C (Hafizh -> Seno)

Berisi fungsi preprocessing, wrapper DINOv2-Large offline (weights/), dan kontrak `ekstrak_semua`
yang IDENTIK 1:1 dengan 04_v3_extract_pair_dino.ipynb (FP32 deterministik).
Gunakan `ekstrak_semua(data_dir, ids)` di 01_model.ipynb dan 02_submission.ipynb.
"""
import gc
import os
import json
import time
import random
import warnings
warnings.filterwarnings("ignore")

import cv2
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from scipy.ndimage import distance_transform_edt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
try:
    import timm
except ImportError:
    timm = None
try:
    import transformers
except ImportError:
    transformers = None


# ==================== KONFIGURASI TERKUNCI (V4 CONTRACT) ====================
SEED = 42
BACKBONE_CHOICE = "dinov2_large"       # Pilihan: "dinov2_large" (Apache-2.0), "dinov2_base", "dinov3_sat_large"
IMG_SIZE_V3 = 256                      # Kelipatan 16 -> 16x16 = 256 patch untuk DINOv3 SAT-493M
DINOV2_FALLBACK_IMG_SIZE = 308         # Kelipatan 14 -> 22x22 = 484 patch untuk DINOv2-L / DINOv2-B
BATCH_SIZE = 16
NUM_WORKERS = 2
EXTRACT_GRAY3 = True
USE_FP16 = False                       # FP32 deterministik penuh (stick 100% ke keputusan_tim_ringkas.md & aturan pengumpulan)
ALLOW_DATAPARALLEL = False             # Sesuai run Version 5 (1 GPU konsisten dengan 1x Kaggle T4 di 02_submission.ipynb)
TINT_NORM_THR = 0.05                   # Pure asymmetric max(0, min(R-G, B-G))
SHARED_CROP_MIN_RATIO = 0.50           # Side shared jika min/max >= 0.50
CLOUD_PATCH_THR = 0.50                 # Maksimal fraksi awan per patch untuk pooling post/changed
# ===========================================================================


def set_reproducible_seed(seed: int = SEED):
    """Kunci RNG dan flag deterministik PyTorch/cuDNN agar rerun konsisten."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def get_hf_token():
    tok = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_TOKEN")
    if tok:
        return tok.strip()
    try:
        from kaggle_secrets import UserSecretsClient
        tok = UserSecretsClient().get_secret("HF_TOKEN")
        if tok:
            return tok.strip()
    except Exception:
        pass
    try:
        from google.colab import userdata
        tok = userdata.get("HF_TOKEN")
        if tok:
            return tok.strip()
    except Exception:
        pass
    return None


def detect_valid_mask(rgb: np.ndarray, sum_thr: int = 10) -> np.ndarray:
    """Deteksi area piksel non-hitam valid."""
    return rgb.sum(axis=2) > sum_thr


def _extract_valid_crop(rgb: np.ndarray):
    valid_raw = detect_valid_mask(rgb)
    if int(valid_raw.sum()) < 50:
        return rgb.copy(), np.ones(rgb.shape[:2], dtype=bool), max(rgb.shape[:2]), False
    ys, xs = np.where(valid_raw)
    y0, y1 = max(0, int(ys.min()) - 4), min(rgb.shape[0], int(ys.max()) + 5)
    x0, x1 = max(0, int(xs.min()) - 4), min(rgb.shape[1], int(xs.max()) + 5)
    crop = rgb[y0:y1, x0:x1].copy()
    m_crop = valid_raw[y0:y1, x0:x1]
    side = max(crop.shape[0], crop.shape[1])
    return crop, m_crop, side, True


def _pad_inpaint_resize(crop: np.ndarray, m_crop: np.ndarray, side: int, out_size: int):
    h, w = crop.shape[:2]
    side = max(side, h, w)
    py, px = (side - h) // 2, (side - w) // 2
    sq_img = np.zeros((side, side, 3), dtype=np.uint8)
    sq_msk = np.zeros((side, side), dtype=bool)
    sq_img[py:py + h, px:px + w] = crop
    sq_msk[py:py + h, px:px + w] = m_crop

    if not sq_msk.all() and sq_msk.any():
        _, idx = distance_transform_edt(~sq_msk, return_indices=True)
        filled = sq_img.copy()
        filled[~sq_msk] = sq_img[idx[0][~sq_msk], idx[1][~sq_msk]]
        blurred = cv2.GaussianBlur(filled, (15, 15), 0)
        sq_img[~sq_msk] = blurred[~sq_msk]

    interp = cv2.INTER_AREA if side >= out_size else cv2.INTER_LINEAR
    return (
        cv2.resize(sq_img, (out_size, out_size), interpolation=interp),
        cv2.resize(sq_msk.astype(np.float32), (out_size, out_size), interpolation=cv2.INTER_AREA),
    )


def crop_pad_inpaint(rgb: np.ndarray, out_size: int, force_side: int = None):
    """Crop ketat bounding box valid, pad persegi (opsional shared side), dan inpaint tepi kosong."""
    crop, m_crop, side, ok = _extract_valid_crop(rgb)
    if not ok:
        return (
            cv2.resize(crop, (out_size, out_size), interpolation=cv2.INTER_AREA),
            np.ones((out_size, out_size), dtype=np.float32),
        )
    use_side = side if force_side is None else max(force_side, side)
    return _pad_inpaint_resize(crop, m_crop, use_side, out_size)


def crop_pad_inpaint_pair(rgb_pre: np.ndarray, rgb_post: np.ndarray, out_size: int):
    """Gunakan side_shared jika rasio skala pre/post >= 0.50 agar resolusi spasial seragam."""
    cp, mp, sp, ok_p = _extract_valid_crop(rgb_pre)
    cq, mq, sq, ok_q = _extract_valid_crop(rgb_post)
    if not ok_p or not ok_q:
        return crop_pad_inpaint(rgb_pre, out_size), crop_pad_inpaint(rgb_post, out_size), False

    ratio = min(sp, sq) / float(max(sp, sq) + 1e-6)
    if ratio >= SHARED_CROP_MIN_RATIO:
        side_shared = min(1024, max(sp, sq))
        return _pad_inpaint_resize(cp, mp, side_shared, out_size), _pad_inpaint_resize(cq, mq, side_shared, out_size), True
    return _pad_inpaint_resize(cp, mp, sp, out_size), _pad_inpaint_resize(cq, mq, sq, out_size), False


def selective_color_norm(rgb_sq: np.ndarray, msk_sq: np.ndarray, tint_thr: float = TINT_NORM_THR) -> np.ndarray:
    """Detektor tint asimetris murni (R > G AND B > G) pada skala [0, 255]; pertahankan hutan/laut alami."""
    arr = rgb_sq.astype(np.float32)
    v = msk_sq > 0.5
    if int(v.sum()) < 30:
        v = np.ones_like(msk_sq, dtype=bool)
    rm, gm, bm = arr[:, :, 0][v].mean(), arr[:, :, 1][v].mean(), arr[:, :, 2][v].mean()
    tint = max(0.0, min(float(rm - gm), float(bm - gm))) / 255.0
    if tint > tint_thr:
        mu = np.array([rm, gm, bm], dtype=np.float32)[None, None, :]
        sd = np.maximum(np.array([arr[:, :, c][v].std() for c in range(3)], dtype=np.float32), 12.0)[None, None, :]
        return np.clip(127.5 + 48.0 * ((arr - mu) / sd), 0.0, 255.0).astype(np.uint8)
    return rgb_sq


def make_gray3_clahe(rgb_sq: np.ndarray) -> np.ndarray:
    """Cabang grayscale CLAHE 3-channel yang kebal acak warna/tint."""
    g = cv2.cvtColor(rgb_sq, cv2.COLOR_RGB2GRAY)
    cg = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(g)
    return np.stack([cg, cg, cg], axis=-1)


class PatchTokenWrapper(nn.Module):
    """Ekstrak patch spasial murni dengan membuang prefix token (CLS + register tokens) secara dinamis."""
    def __init__(self, core_model, is_hf: bool, n_prefix: int):
        super().__init__()
        self.core_model = core_model
        self.is_hf = is_hf
        self.n_prefix = n_prefix

    def forward(self, x):
        if self.is_hf:
            tokens = self.core_model(pixel_values=x).last_hidden_state
        else:
            tokens = self.core_model.forward_features(x)
        return tokens[:, self.n_prefix:, :]


def _has_hf_weights(d: str) -> bool:
    return bool(d) and os.path.isdir(d) and os.path.exists(os.path.join(d, "config.json")) and (
        os.path.exists(os.path.join(d, "model.safetensors")) or os.path.exists(os.path.join(d, "pytorch_model.bin"))
    )


def _has_timm_weights(d: str) -> bool:
    return bool(d) and os.path.isdir(d) and os.path.exists(os.path.join(d, "model_state.pt")) and os.path.exists(os.path.join(d, "backbone_config.json"))


def find_local_weights_dir(backbone_choice: str = BACKBONE_CHOICE, explicit_dir: str = None):
    """Cari folder bobot lokal di ./weights, /kaggle/working/weights, atau /kaggle/input/ agar bisa jalan offline."""
    candidates = []
    if explicit_dir:
        candidates.append(explicit_dir)
    for base in ["./weights", "../weights", "/kaggle/working/weights", "/kaggle/input"]:
        if not os.path.exists(base):
            continue
        candidates.append(os.path.join(base, backbone_choice))
        if base == "/kaggle/input":
            for root, dirs, _ in os.walk(base):
                if os.path.basename(root) == backbone_choice:
                    candidates.append(root)
                if "weights" in dirs:
                    candidates.append(os.path.join(root, "weights", backbone_choice))
    for cand in candidates:
        if _has_hf_weights(cand) or _has_timm_weights(cand):
            return cand
    return None


def _build_meta_info(base_info: dict, device: str, loaded_from: str):
    gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"
    base_info.update({
        "seed": SEED,
        "tint_norm_thr": TINT_NORM_THR,
        "cloud_pool_thr": CLOUD_PATCH_THR,
        "shared_crop_min_ratio": SHARED_CROP_MIN_RATIO,
        "precision": "fp16_autocast" if (USE_FP16 and device == "cuda") else "fp32",
        "device": device,
        "gpu_name": gpu_name,
        "n_gpus_visible": int(torch.cuda.device_count()) if torch.cuda.is_available() else 0,
        "loaded_from": loaded_from,
        "torch_version": torch.__version__,
        "transformers_version": getattr(transformers, "__version__", "not_installed"),
        "timm_version": getattr(timm, "__version__", "not_installed"),
    })
    return base_info


def _save_backbone_meta(save_dir: str, info: dict):
    os.makedirs(save_dir, exist_ok=True)
    with open(os.path.join(save_dir, "backbone_config.json"), "w", encoding="utf-8") as f:
        json.dump(info, f, indent=2)


def load_dino_backbone(
    backbone_choice: str = BACKBONE_CHOICE,
    device: str = None,
    hf_token: str = None,
    local_weights_dir: str = None,
    save_weights_dir: str = None,
):
    """Muat backbone secara offline dari weights/ bila tersedia; jika online, download lalu simpan ke save_weights_dir."""
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"
    if hf_token is None:
        hf_token = get_hf_token()

    found_local = find_local_weights_dir(backbone_choice, explicit_dir=local_weights_dir)
    if found_local:
        if _has_hf_weights(found_local):
            from transformers import AutoImageProcessor, AutoModel
            proc = AutoImageProcessor.from_pretrained(found_local, local_files_only=True)
            hf_m = AutoModel.from_pretrained(found_local, local_files_only=True).eval().to(device)
            n_pref = 1 + int(getattr(hf_m.config, "num_register_tokens", 4 if "dinov3" in backbone_choice else 0))
            p_sz = int(getattr(hf_m.config, "patch_size", 16 if "dinov3" in backbone_choice else 14))
            img_sz = IMG_SIZE_V3 if "dinov3" in backbone_choice else DINOV2_FALLBACK_IMG_SIZE
            cfg_p = os.path.join(found_local, "backbone_config.json")
            saved_cfg = json.load(open(cfg_p, "r", encoding="utf-8")) if os.path.exists(cfg_p) else {}
            info = _build_meta_info({
                "model": saved_cfg.get("model", found_local),
                "backbone_choice": backbone_choice,
                "patch_size": int(saved_cfg.get("patch_size", p_sz)),
                "n_prefix": int(saved_cfg.get("n_prefix", n_pref)),
                "img_size": int(saved_cfg.get("img_size", img_sz)),
                "dim": int(hf_m.config.hidden_size),
                "mean": list(getattr(proc, "image_mean", saved_cfg.get("mean", [0.430, 0.411, 0.296]))),
                "std": list(getattr(proc, "image_std", saved_cfg.get("std", [0.213, 0.156, 0.143]))),
                "license": saved_cfg.get("license", "DINOv3 License" if "dinov3" in backbone_choice else "Apache-2.0"),
            }, device=device, loaded_from=f"offline:{found_local}")
            print(f"[OFFLINE OK] Model dimuat lokal dari {found_local}")
            return PatchTokenWrapper(hf_m, True, info["n_prefix"]), info

        if _has_timm_weights(found_local):
            saved_cfg = json.load(open(os.path.join(found_local, "backbone_config.json"), "r", encoding="utf-8"))
            m = timm.create_model(saved_cfg["model"], pretrained=False, num_classes=0, img_size=saved_cfg["img_size"]).eval()
            state = torch.load(os.path.join(found_local, "model_state.pt"), map_location="cpu")
            m.load_state_dict(state)
            m = m.to(device)
            info = _build_meta_info(saved_cfg, device=device, loaded_from=f"offline:{found_local}")
            print(f"[OFFLINE OK] Model timm dimuat lokal dari {found_local}")
            return PatchTokenWrapper(m, False, info["n_prefix"]), info

    if backbone_choice == "dinov3_sat_large":
        try:
            from transformers import AutoImageProcessor, AutoModel
            repo = "facebook/dinov3-vitl16-pretrain-sat493m"
            proc = AutoImageProcessor.from_pretrained(repo, token=hf_token)
            hf_m = AutoModel.from_pretrained(repo, token=hf_token).eval().to(device)
            n_pref = 1 + int(getattr(hf_m.config, "num_register_tokens", 4))
            p_sz = int(getattr(hf_m.config, "patch_size", 16))
            info = _build_meta_info({
                "model": repo,
                "backbone_choice": "dinov3_sat_large",
                "patch_size": p_sz,
                "n_prefix": n_pref,
                "img_size": IMG_SIZE_V3,
                "dim": int(hf_m.config.hidden_size),
                "mean": list(getattr(proc, "image_mean", [0.430, 0.411, 0.296])),
                "std": list(getattr(proc, "image_std", [0.213, 0.156, 0.143])),
                "license": "DINOv3 License",
            }, device=device, loaded_from=f"online:{repo}")
            if save_weights_dir:
                out_w = os.path.join(save_weights_dir, "dinov3_sat_large")
                os.makedirs(out_w, exist_ok=True)
                proc.save_pretrained(out_w)
                hf_m.save_pretrained(out_w)
                _save_backbone_meta(out_w, info)
                print(f"[SAVED WEIGHTS] Bobot DINOv3 disimpan untuk offline: {out_w}")
            return PatchTokenWrapper(hf_m, True, n_pref), info
        except Exception as e:
            print(f"[INFO] Fallback ke DINOv2 karena: {type(e).__name__}: {str(e)[:160]}")

    if backbone_choice == "dinov2_base":
        from transformers import AutoImageProcessor, AutoModel
        repo = "facebook/dinov2-base"
        proc = AutoImageProcessor.from_pretrained(repo)
        hf_m = AutoModel.from_pretrained(repo).eval().to(device)
        n_pref = 1 + int(getattr(hf_m.config, "num_register_tokens", 0))
        info = _build_meta_info({
            "model": repo,
            "backbone_choice": "dinov2_base",
            "patch_size": 14,
            "n_prefix": n_pref,
            "img_size": DINOV2_FALLBACK_IMG_SIZE,
            "dim": int(hf_m.config.hidden_size),
            "mean": list(getattr(proc, "image_mean", [0.485, 0.456, 0.406])),
            "std": list(getattr(proc, "image_std", [0.229, 0.224, 0.225])),
            "license": "Apache-2.0",
        }, device=device, loaded_from=f"online:{repo}")
        if save_weights_dir:
            out_w = os.path.join(save_weights_dir, "dinov2_base")
            os.makedirs(out_w, exist_ok=True)
            proc.save_pretrained(out_w)
            hf_m.save_pretrained(out_w)
            _save_backbone_meta(out_w, info)
            print(f"[SAVED WEIGHTS] Bobot DINOv2-Base disimpan untuk offline: {out_w}")
        return PatchTokenWrapper(hf_m, True, n_pref), info

    name = "vit_large_patch14_dinov2.lvd142m"
    m = timm.create_model(name, pretrained=True, num_classes=0, img_size=DINOV2_FALLBACK_IMG_SIZE).eval().to(device)
    n_pref = int(getattr(m, "num_prefix_tokens", 1))
    info = _build_meta_info({
        "model": name,
        "backbone_choice": "dinov2_large",
        "patch_size": 14,
        "n_prefix": n_pref,
        "img_size": DINOV2_FALLBACK_IMG_SIZE,
        "dim": int(m.num_features),
        "mean": [0.485, 0.456, 0.406],
        "std": [0.229, 0.224, 0.225],
        "license": "Apache-2.0",
    }, device=device, loaded_from=f"online:{name}")
    if save_weights_dir:
        out_w = os.path.join(save_weights_dir, "dinov2_large")
        os.makedirs(out_w, exist_ok=True)
        torch.save(m.state_dict(), os.path.join(out_w, "model_state.pt"))
        _save_backbone_meta(out_w, info)
        print(f"[SAVED WEIGHTS] Bobot DINOv2-Large disimpan untuk offline: {out_w}")
    return PatchTokenWrapper(m, False, n_pref), info


def d4_augment_batch(x: torch.Tensor, m: torch.Tensor, fl: bool, rot: int):
    """Terapkan salah satu dari 8 transformasi simetri D4 secara sinkron pada gambar & mask."""
    xf = torch.flip(x, [3]) if fl else x
    mf = torch.flip(m.float(), [2]) > 0.5 if fl else m
    xv = torch.rot90(xf, rot, [2, 3]) if rot > 0 else xf
    mv = torch.rot90(mf, rot, [1, 2]) if rot > 0 else mf
    return xv, mv


def extract_patch_grid(model, x_v: torch.Tensor, m_v: torch.Tensor, device: str = None, use_fp16: bool = USE_FP16):
    """Jalankan forward lalu kembalikan patch L2-normalized & pooled embedding ter-mask."""
    if device is None:
        device = str(x_v.device).split(":")[0]
    bs = x_v.shape[0]
    amp_enabled = bool(use_fp16 and device == "cuda")
    with torch.amp.autocast("cuda", enabled=amp_enabled, dtype=torch.float16):
        toks = model(x_v)
    toks_n = F.normalize(toks.float(), dim=2)
    mw = m_v.reshape(bs, -1, 1).float()
    pooled = F.normalize((toks_n * mw).sum(dim=1) / (mw.sum(dim=1) + 1e-6), dim=1)
    return toks_n, mw, pooled


def compute_spatial_stats(cmap: np.ndarray, vmask: np.ndarray, grid_sz: int):
    """Statistik spasial perubahan patch bebas rotasi berbasis P10 + P15 + floor absolut."""
    out = {}
    keys = (
        "frac_chg_abs", "frac_chg_p10", "chg_mean_p10", "chg_p90_p10",
        "frac_chg_p15", "chg_mean_p15", "chg_p90_p15", "massive_damage_guard",
        "spatial_entropy", "top10_share", "radial_decay", "com_shift", "axis_skew",
    )
    if int(vmask.sum()) < 4:
        for k in keys:
            out[k] = 0.0
        return out, np.ones(grid_sz * grid_sz, dtype=np.float32) / float(grid_sz * grid_sz)

    vals = cmap[vmask]
    p10 = float(np.percentile(vals, 10))
    p15 = float(np.percentile(vals, 15))
    crel10 = np.maximum(cmap - p10, 0.0) * vmask.astype(np.float32)
    crel15 = np.maximum(cmap - p15, 0.0) * vmask.astype(np.float32)
    vrel10 = crel10[vmask]
    vrel15 = crel15[vmask]

    out["frac_chg_abs"] = float(np.mean(vals > 0.25))
    out["frac_chg_p10"] = float(np.mean(vrel10 > 0.12))
    out["chg_mean_p10"] = float(vrel10.mean())
    out["chg_p90_p10"] = float(np.percentile(vrel10, 90))
    out["frac_chg_p15"] = float(np.mean(vrel15 > 0.12))
    out["chg_mean_p15"] = float(vrel15.mean())
    out["chg_p90_p15"] = float(np.percentile(vrel15, 90))
    out["massive_damage_guard"] = float((out["frac_chg_p15"] < 0.05) and (out["frac_chg_abs"] > 0.50))

    if out["massive_damage_guard"] > 0.5:
        crel_use = np.maximum(cmap - 0.10, 0.0) * vmask.astype(np.float32)
    else:
        crel_use = crel15
    vrel = crel_use[vmask]
    total = float(vrel.sum()) + 1e-8
    probs = vrel / total

    out["spatial_entropy"] = float(-np.sum(probs * np.log(probs + 1e-12)) / (np.log(len(probs) + 1e-12) + 1e-8))
    k10 = max(1, int(0.10 * len(vrel)))
    out["top10_share"] = float(np.sort(vrel)[-k10:].sum() / total)

    ys, xs = np.mgrid[0:grid_sz, 0:grid_sz].astype(np.float32)
    yv, xv = ys[vmask], xs[vmask]
    cy, cx = float(np.sum(probs * yv)), float(np.sum(probs * xv))
    out["com_shift"] = float(np.hypot(cy - (grid_sz - 1) / 2.0, cx - (grid_sz - 1) / 2.0) / (grid_sz * 0.5))

    pk = int(np.argmax(crel_use))
    py, px = float(pk // grid_sz), float(pk % grid_sz)
    dist = np.hypot(yv - py, xv - px)
    out["radial_decay"] = float(-np.corrcoef(dist, vrel)[0, 1]) if dist.std() > 1e-6 and vrel.std() > 1e-6 else 0.0

    coords = np.stack([xv - cx, yv - cy], axis=1)
    cov = (coords * probs[:, None]).T @ coords
    _, eigvecs = np.linalg.eigh(cov)
    proj = coords @ eigvecs[:, -1]
    sd_p = float(np.sqrt(np.sum(probs * (proj ** 2))) + 1e-6)
    out["axis_skew"] = float(abs(np.sum(probs * (proj ** 3)) / (sd_p ** 3)))

    weights = crel_use.reshape(-1)
    wsum = float(weights.sum())
    weights = (weights / wsum) if wsum > 1e-8 else (vmask.reshape(-1).astype(np.float32) / float(vmask.sum()))
    return out, weights.astype(np.float32)


class PairDataset(Dataset):
    def __init__(self, df: pd.DataFrame, img_size: int, mean, std, mode: str = "color_norm"):
        self.df = df.reset_index(drop=True)
        self.img_size = img_size
        self.mode = mode
        self.mean = np.array(mean, dtype=np.float32)[:, None, None]
        self.std = np.array(std, dtype=np.float32)[:, None, None]

    def __len__(self):
        return len(self.df)

    def _to_tensor_and_cloud(self, sq_img: np.ndarray, sq_msk: np.ndarray):
        view = make_gray3_clahe(sq_img) if self.mode == "gray3" else selective_color_norm(sq_img, sq_msk)
        arr = sq_img.astype(np.float32) / 255.0
        bright = arr.mean(axis=2)
        cmax, cmin = arr.max(axis=2), arr.min(axis=2)
        sat = (cmax - cmin) / (cmax + 1e-5)
        cld = ((bright > 0.85) & (sat < 0.14)).astype(np.float32)

        chw = ((view.transpose(2, 0, 1).astype(np.float32) / 255.0) - self.mean) / self.std
        chw[:, sq_msk <= 0.5] = 0.0
        return torch.from_numpy(chw), torch.from_numpy(sq_msk), torch.from_numpy(cld)

    def __getitem__(self, idx):
        r = self.df.iloc[idx]
        bgr_p, bgr_q = cv2.imread(r["pre_path"]), cv2.imread(r["post_path"])
        if bgr_p is None or bgr_q is None:
            raise FileNotFoundError(f"Gagal membaca gambar: {r['Id']}")
        (sq_p, m_p), (sq_q, m_q), _ = crop_pad_inpaint_pair(
            cv2.cvtColor(bgr_p, cv2.COLOR_BGR2RGB),
            cv2.cvtColor(bgr_q, cv2.COLOR_BGR2RGB),
            self.img_size,
        )
        xp, mp, _ = self._to_tensor_and_cloud(sq_p, m_p)
        xq, mq, cq = self._to_tensor_and_cloud(sq_q, m_q)
        return xp, mp, xq, mq, cq


def select_cloud_excluded_mask(mq_grid: torch.Tensor, cq_grid: torch.Tensor):
    """Pool post/changed di atas patch bebas awan (<= CLOUD_PATCH_THR); fallback all_valid hanya jika <4 patch bersih."""
    vq_all = mq_grid >= 0.5
    vq_clean = vq_all & (cq_grid <= CLOUD_PATCH_THR)
    clean_cnt = vq_clean.reshape(vq_clean.shape[0], -1).sum(dim=1)
    fallback_flag = clean_cnt < 4
    vq_pool = torch.where(fallback_flag[:, None, None], vq_all, vq_clean)
    return vq_pool, vq_all, fallback_flag


def run_pairwise_extraction(
    model,
    df: pd.DataFrame,
    b_info: dict,
    mode: str = "color_norm",
    compute_patch_stats: bool = True,
    device: str = None,
    batch_size: int = BATCH_SIZE,
    num_workers: int = NUM_WORKERS,
    use_fp16: bool = USE_FP16,
):
    if device is None:
        device = b_info.get("device", "cuda" if torch.cuda.is_available() else "cpu")
    img_sz, p_sz, emb_dim = b_info["img_size"], b_info["patch_size"], b_info["dim"]
    g_sz = img_sz // p_sz
    ds = PairDataset(df, img_sz, b_info["mean"], b_info["std"], mode=mode)
    dl = DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=(device == "cuda"))

    N = len(df)
    ids = df["Id"].astype(str).tolist()
    emb_pre = np.zeros((N, emb_dim), dtype=np.float32)
    emb_post = np.zeros((N, emb_dim), dtype=np.float32)
    emb_chg = np.zeros((N, emb_dim), dtype=np.float32) if compute_patch_stats else None
    pstats = []
    ptr = 0

    with torch.no_grad():
        for xp, mp, xq, mq, cq in tqdm(dl, desc=f"Extract [{mode}]"):
            bs = xp.shape[0]
            xp, mp = xp.to(device, non_blocking=True), mp.to(device, non_blocking=True)
            xq, mq, cq = xq.to(device, non_blocking=True), mq.to(device, non_blocking=True), cq.to(device, non_blocking=True)

            vp_base = F.avg_pool2d(mp.unsqueeze(1), p_sz, p_sz).squeeze(1) >= 0.5
            mq_grid = F.avg_pool2d(mq.unsqueeze(1), p_sz, p_sz).squeeze(1)
            cq_grid = F.avg_pool2d(cq.unsqueeze(1), p_sz, p_sz).squeeze(1)
            vq_pool, vq_all, fb_flag = select_cloud_excluded_mask(mq_grid, cq_grid)

            acc_pre = torch.zeros((bs, emb_dim), device=device, dtype=torch.float32)
            acc_post = torch.zeros((bs, emb_dim), device=device, dtype=torch.float32)
            pre_bank = []
            post_tok0 = None

            for fl in (False, True):
                for rot in (0, 1, 2, 3):
                    xp_v, mp_v = d4_augment_batch(xp, vp_base, fl, rot)
                    xq_v, mq_v = d4_augment_batch(xq, vq_pool, fl, rot)

                    tp_n, mp_w, gp = extract_patch_grid(model, xp_v, mp_v, device=device, use_fp16=use_fp16)
                    tq_n, _, gq = extract_patch_grid(model, xq_v, mq_v, device=device, use_fp16=use_fp16)
                    acc_pre += gp
                    acc_post += gq

                    if compute_patch_stats:
                        pre_bank.append(tp_n * mp_w)
                        if not fl and rot == 0:
                            post_tok0 = tq_n

            emb_pre[ptr:ptr + bs] = F.normalize(acc_pre, dim=1).cpu().numpy()
            emb_post[ptr:ptr + bs] = F.normalize(acc_post, dim=1).cpu().numpy()

            if compute_patch_stats:
                bank_all = torch.cat(pre_bank, dim=1)
                sim = torch.bmm(post_tok0, bank_all.transpose(1, 2))
                max_sim, _ = sim.max(dim=2)
                chg_map = (1.0 - max_sim).reshape(bs, g_sz, g_sz).cpu().numpy()
                vp_np = vq_pool.cpu().numpy()
                va_np = vq_all.cpu().numpy()
                fb_np = fb_flag.cpu().numpy()
                pt_np = post_tok0.cpu().numpy()
                for i in range(bs):
                    st, w_chg = compute_spatial_stats(chg_map[i], vp_np[i], g_sz)
                    st_all, _ = compute_spatial_stats(chg_map[i], va_np[i], g_sz)
                    st["cloud_excluded_fallback_flag"] = float(fb_np[i])
                    for k_a, v_a in st_all.items():
                        st[f"{k_a}_allvalid"] = v_a
                    st["Id"] = ids[ptr + i]
                    pstats.append(st)
                    vec = (pt_np[i] * w_chg[:, None]).sum(axis=0)
                    emb_chg[ptr + i] = vec / (np.linalg.norm(vec) + 1e-8)

            ptr += bs

    return emb_pre, emb_post, emb_chg, pd.DataFrame(pstats) if compute_patch_stats else None


def compute_tes_a_retrieval(pre_emb: np.ndarray, post_emb: np.ndarray, ks=(1, 5, 10)) -> dict:
    """Tes A: pencocokan post -> pre per pasangan (tanpa label), mengukur ketahanan preprocessing."""
    n = pre_emb.shape[0]
    p_norm = pre_emb / (np.linalg.norm(pre_emb, axis=1, keepdims=True) + 1e-8)
    q_norm = post_emb / (np.linalg.norm(post_emb, axis=1, keepdims=True) + 1e-8)
    sim = q_norm @ p_norm.T
    ranks = np.argsort(-sim, axis=1)
    targets = np.arange(n)[:, None]
    out = {}
    for k in ks:
        k_use = min(int(k), n)
        out[f"Recall@{k}"] = float(np.mean(np.any(ranks[:, :k_use] == targets, axis=1)))
    return out

def compute_artifact_and_phys(rgb_raw: np.ndarray) -> dict:
    """Ekstraksi fitur artifak & sinyal fisik bencana per citra (skala [0, 1] untuk pink_cast)."""
    small = cv2.resize(rgb_raw, (256, 256), interpolation=cv2.INTER_AREA)
    valid = detect_valid_mask(small)
    if int(valid.sum()) < 50:
        valid = np.ones((256, 256), dtype=bool)
    arr = small.astype(np.float32) / 255.0
    rv, gv, bv = arr[:, :, 0][valid], arr[:, :, 1][valid], arr[:, :, 2][valid]
    bright = (rv + gv + bv) / 3.0
    cmax, cmin = np.maximum(np.maximum(rv, gv), bv), np.minimum(np.minimum(rv, gv), bv)
    sat = (cmax - cmin) / (cmax + 1e-5)

    pink_cast = float(max(0.0, min(float(rv.mean() - gv.mean()), float(bv.mean() - gv.mean()))))
    chroma_std = float(np.mean(np.std(arr[valid], axis=1)))
    gray_u8 = cv2.cvtColor(small, cv2.COLOR_RGB2GRAY)
    gy = np.abs(cv2.Sobel(gray_u8, cv2.CV_32F, 0, 1))
    gx = np.abs(cv2.Sobel(gray_u8, cv2.CV_32F, 1, 0))
    lap = float(cv2.Laplacian(gray_u8, cv2.CV_32F)[valid].var())

    return {
        "black_frac": float(1.0 - valid.mean()),
        "cloud_frac": float(np.mean((bright > 0.85) & (sat < 0.14))),
        "tint_score": pink_cast,
        "is_gray": float(chroma_std < 0.02),
        "chroma_std": chroma_std,
        "brightness": float(bright.mean()),
        "sat": float(sat.mean()),
        "blur_score": lap,
        "scanline_score": float((gy[valid].mean() + 1e-5) / (gx[valid].mean() + 1e-5)),
        "exg": float(np.mean(2.0 * gv - rv - bv)),
        "water": float(np.mean((bv > rv + 0.03) & (bv > gv + 0.01) & (bright < 0.65))),
        "dark_char": float(np.mean(bright < 0.22)),
        "ash_ratio": float(np.mean((sat < 0.14) & (bright >= 0.25) & (bright <= 0.72))),
        "edge": float(np.hypot(gx, gy)[valid].mean() / 255.0),
    }

try:
    from skimage.filters.rank import entropy as _rank_entropy
    from skimage.morphology import disk as _disk
    HAS_SKIMAGE = True
except ImportError:
    HAS_SKIMAGE = False


# ==================== KONFIGURASI PHYS V4 ====================
AMBANG_COLOR_SHIFT = 24.0        # Sesuai run Version 5 (probe 26.2 & 29.8 tertangkap, hazing tetap ketat)
AMBANG_HAZE_STD = 25.0           # std(gray) < 25
AMBANG_HAZE_MEAN = 150.0         # mean(gray) > 150
BIRU_REL_BG_THR = 12.0           # (B-G) - median(B-G) > 12 (skala 0-255)
BIRU_REL_BR_THR = 12.0           # (B-R) - median(B-R) > 12 (skala 0-255)
WATER_TEX_THR = 18.0             # Tekstur gradient rendah untuk piksel air relatif
# ============================================================


def _local_entropy_mean(gray_u8: np.ndarray, valid: np.ndarray) -> float:
    """Hitung rata-rata entropi lokal pada grayscale CLAHE (hanya di piksel valid)."""
    clahe_g = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(gray_u8)
    if HAS_SKIMAGE:
        ent = _rank_entropy(clahe_g, _disk(5)).astype(np.float32)
        return float(ent[valid].mean())
    # Fallback cepat berbasis varians lokal ternormalisasi log bila scikit-image tidak tersedia
    blur1 = cv2.GaussianBlur(clahe_g.astype(np.float32), (11, 11), 0)
    blur2 = cv2.GaussianBlur((clahe_g.astype(np.float32) ** 2), (11, 11), 0)
    lstd = np.sqrt(np.maximum(blur2 - blur1 ** 2, 1.0))
    return float(np.log2(lstd[valid] + 1.0).mean())


def _extract_single_phys_v4(rgb_raw: np.ndarray, tint_thr: float = TINT_NORM_THR) -> dict:
    small = cv2.resize(rgb_raw, (256, 256), interpolation=cv2.INTER_AREA)
    valid = detect_valid_mask(small)
    if int(valid.sum()) < 50:
        valid = np.ones((256, 256), dtype=bool)

    arr255 = small.astype(np.float32)
    arr01 = arr255 / 255.0
    rv255, gv255, bv255 = arr255[:, :, 0][valid], arr255[:, :, 1][valid], arr255[:, :, 2][valid]
    rv, gv, bv = arr01[:, :, 0][valid], arr01[:, :, 1][valid], arr01[:, :, 2][valid]

    gray_u8 = cv2.cvtColor(small, cv2.COLOR_RGB2GRAY)
    gv_u8 = gray_u8[valid].astype(np.float32)
    med_gray = float(np.median(gv_u8))
    dark_rel = float(np.mean(gv_u8 < (0.5 * med_gray)))

    rg = float(np.mean(rv255 - gv255))
    bg = float(np.mean(bv255 - gv255))
    tint_pink = float(max(0.0, min(float(rv.mean() - gv.mean()), float(bv.mean() - gv.mean()))))
    chroma_std = float(np.mean(np.std(arr01[valid], axis=1)))
    is_gray = float(chroma_std < 0.02)
    haze_flag = float((float(gv_u8.std()) < AMBANG_HAZE_STD) and (float(gv_u8.mean()) > AMBANG_HAZE_MEAN))

    # Air relatif terhadap median gambar sendiri + tekstur gradient rendah
    bg_diff = arr255[:, :, 2] - arr255[:, :, 1]
    br_diff = arr255[:, :, 2] - arr255[:, :, 0]
    med_bg = float(np.median(bg_diff[valid]))
    med_br = float(np.median(br_diff[valid]))
    gx = np.abs(cv2.Sobel(gray_u8, cv2.CV_32F, 1, 0))
    gy = np.abs(cv2.Sobel(gray_u8, cv2.CV_32F, 0, 1))
    grad = np.hypot(gx, gy)
    bright_px = arr01.mean(axis=2)
    water_mask = (
        ((bg_diff - med_bg) > BIRU_REL_BG_THR)
        & ((br_diff - med_br) > BIRU_REL_BR_THR)
        & (grad < WATER_TEX_THR)
        & (bright_px < 0.70)
    )
    water_rel = float(water_mask[valid].mean())

    cmax, cmin = np.maximum(np.maximum(rv, gv), bv), np.minimum(np.minimum(rv, gv), bv)
    sat = (cmax - cmin) / (cmax + 1e-5)
    bright_v = (rv + gv + bv) / 3.0

    return {
        "rg": rg,
        "bg": bg,
        "tint_pink": float(tint_pink > tint_thr),
        "is_gray": is_gray,
        "haze_flag": haze_flag,
        "dark_rel": dark_rel,
        "water_rel": water_rel,
        "entropy": _local_entropy_mean(gray_u8, valid),
        "exg": float(np.mean(2.0 * gv - rv - bv)),
        "ash_ratio": float(np.mean((sat < 0.14) & (bright_v >= 0.25) & (bright_v <= 0.72))),
        "sat": float(sat.mean()),
        "brightness": float(bright_v.mean()),
    }


def compute_phys_v4_pair(uid: str, rgb_pre: np.ndarray, rgb_post: np.ndarray) -> dict:
    """Hitung 23 kolom phys_v4.parquet sesuai perubahan_preprocessing_v4.md."""
    sp = _extract_single_phys_v4(rgb_pre)
    sq = _extract_single_phys_v4(rgb_post)

    color_shift = float(np.hypot(sq["rg"] - sp["rg"], sq["bg"] - sp["bg"]))
    color_shift_flag = float(color_shift > AMBANG_COLOR_SHIFT)
    unrel = bool(
        (sp["is_gray"] > 0.5)
        or (sq["is_gray"] > 0.5)
        or (sp["tint_pink"] > 0.5)
        or (sq["tint_pink"] > 0.5)
        or (color_shift_flag > 0.5)
        or (sp["haze_flag"] > 0.5)
        or (sq["haze_flag"] > 0.5)
    )

    water_rel_pre = np.nan if unrel else sp["water_rel"]
    water_rel_post = np.nan if unrel else sq["water_rel"]
    d_water = np.nan if unrel else (sq["water_rel"] - sp["water_rel"])

    return {
        "Id": uid,
        "rg_pre": sp["rg"],
        "bg_pre": sp["bg"],
        "rg_post": sq["rg"],
        "bg_post": sq["bg"],
        "color_shift": color_shift,
        "color_shift_flag": color_shift_flag,
        "haze_flag_pre": sp["haze_flag"],
        "haze_flag_post": sq["haze_flag"],
        "color_unreliable_v4": float(unrel),
        "dark_rel_pre": sp["dark_rel"],
        "dark_rel_post": sq["dark_rel"],
        "d_dark": sq["dark_rel"] - sp["dark_rel"],
        "water_rel_pre": water_rel_pre,
        "water_rel_post": water_rel_post,
        "d_water": d_water,
        "entropy_pre": sp["entropy"],
        "entropy_post": sq["entropy"],
        "d_entropy": sq["entropy"] - sp["entropy"],
        "d_exg_v4": np.nan if unrel else (sq["exg"] - sp["exg"]),
        "ash_gain_v4": np.nan if unrel else (sq["ash_ratio"] - sp["ash_ratio"]),
        "sat_drop_v4": np.nan if unrel else (sp["sat"] - sq["sat"]),
        "d_bright_raw_v4": np.nan if unrel else (sq["brightness"] - sp["brightness"]),
        "_raw_d_bright": sq["brightness"] - sp["brightness"],
    }


def audit_phys_v4_dataframe(phys_df: pd.DataFrame, ref_ids: list):
    """Jalankan seluruh pengecekan Bagian 5 perubahan_preprocessing_v4.md."""
    assert list(phys_df["Id"]) == ref_ids, "Urutan Id phys_v4.parquet tidak cocok dengan sample_submission.csv!"
    raw_db = phys_df["_raw_d_bright"].values
    d_dark = phys_df["d_dark"].values
    corr_dark = float(np.corrcoef(d_dark, raw_db)[0, 1]) if raw_db.std() > 1e-8 and d_dark.std() > 1e-8 else 0.0
    unrel_frac = float(phys_df["color_unreliable_v4"].mean())
    print("=== AUDIT PHYS_V4 (BAGIAN 5) ===")
    print(f"Rows: {len(phys_df)} | |corr(d_dark, d_brightness_raw)| = {abs(corr_dark):.4f} (Target < 0.30)")
    print(f"color_unreliable_v4: {int(phys_df['color_unreliable_v4'].sum())}/{len(phys_df)} ({unrel_frac*100:.1f}%)")
    haze_any = ((phys_df["haze_flag_pre"] > 0.5) | (phys_df["haze_flag_post"] > 0.5)).mean() * 100.0
    shift_pct = (phys_df["color_shift_flag"] > 0.5).mean() * 100.0
    print(f"  Breakdown: color_shift_flag={shift_pct:.1f}% | haze_any={haze_any:.1f}% (AMBANG_COLOR_SHIFT={AMBANG_COLOR_SHIFT})")
    if unrel_frac > 0.50:
        print("[PERINGATAN] Porsi color_unreliable_v4 > 50% — pertimbangkan melonggarkan AMBANG_COLOR_SHIFT / AMBANG_HAZE.")
    for probe_id in ("d62c43e0f31c", "63431151b5b5"):
        sub = phys_df[phys_df["Id"] == probe_id]
        if len(sub) > 0:
            r = sub.iloc[0]
            print(f"  Probe {probe_id}: color_unreliable_v4={r['color_unreliable_v4']} | color_shift={r['color_shift']:.1f} | "
                  f"haze=({r['haze_flag_pre']},{r['haze_flag_post']}) | water_rel_pre={r['water_rel_pre']}")
    return corr_dark, unrel_frac

def find_task2_paths():
    for sr in [
        "/kaggle/input/competitions/task-2-datavidia-11-0",
        "/kaggle/input/task-2-datavidia-11-0",
        "/kaggle/input",
        ".",
    ]:
        if not os.path.exists(sr):
            continue
        for root, _, files in os.walk(sr):
            if "sample_submission.csv" in files:
                cand_sub = os.path.join(root, "sample_submission.csv")
                probe_id = str(pd.read_csv(cand_sub, nrows=1)["Id"].iloc[0])
                if os.path.exists(os.path.join(root, probe_id, "pre.jpg")):
                    return root, cand_sub
                if os.path.exists(os.path.join(root, "Fenomena", probe_id, "pre.jpg")):
                    return os.path.join(root, "Fenomena"), cand_sub
    raise FileNotFoundError("Dataset Task-2 (sample_submission.csv + <Id>/pre.jpg) tidak ditemukan.")


def build_pairs_df(data_dir: str, sub_path: str) -> pd.DataFrame:
    """Bangun DataFrame pasangan pre/post sesuai persis urutan Id di sample_submission.csv."""
    all_ids = pd.read_csv(sub_path)["Id"].astype(str).tolist()
    return pd.DataFrame({
        "Id": all_ids,
        "pre_path": [os.path.join(data_dir, uid, "pre.jpg") for uid in all_ids],
        "post_path": [os.path.join(data_dir, uid, "post.jpg") for uid in all_ids],
    })


def ekstrak_semua(
    data_dir,
    ids,
    weights_dir: str = None,
    backbone_choice: str = BACKBONE_CHOICE,
    include_gray: bool = False,
    compute_patch_and_phys: bool = True,
    batch_size: int = BATCH_SIZE,
    require_offline: bool = True,
) -> dict:
    """Kontrak fungsi terkunci A -> C untuk 01_model.ipynb & 02_submission.ipynb (FP32 deterministik)."""
    from pathlib import Path

    set_reproducible_seed(SEED)
    data_root = Path(data_dir)
    id_list = [str(x) for x in ids]
    if len(id_list) == 0:
        raise ValueError("ids kosong!")

    probe = id_list[0]
    if not (data_root / probe / "pre.jpg").exists() and (data_root / "Fenomena" / probe / "pre.jpg").exists():
        data_root = data_root / "Fenomena"

    pairs_df = pd.DataFrame({
        "Id": id_list,
        "pre_path": [str(data_root / uid / "pre.jpg") for uid in id_list],
        "post_path": [str(data_root / uid / "post.jpg") for uid in id_list],
    })

    if require_offline:
        found_w = find_local_weights_dir(backbone_choice, explicit_dir=weights_dir)
        if not found_w:
            raise FileNotFoundError(
                f"Bobot offline '{backbone_choice}' tidak ditemukan. Taruh folder 'weights/{backbone_choice}/' "
                "atau kirim path lewat parameter weights_dir=... (wajib tanpa internet & tanpa HF_TOKEN)."
            )
        weights_dir = found_w

    device = "cuda" if torch.cuda.is_available() else "cpu"
    backbone, b_info = load_dino_backbone(
        backbone_choice=backbone_choice,
        device=device,
        local_weights_dir=weights_dir,
        save_weights_dir=None,
    )
    if ALLOW_DATAPARALLEL and device == "cuda" and torch.cuda.device_count() > 1:
        backbone = nn.DataParallel(backbone)

    nw = 0 if os.name == "nt" else NUM_WORKERS
    c_pre, c_post, c_chg, patch_df = run_pairwise_extraction(
        backbone,
        pairs_df,
        b_info,
        mode="color_norm",
        compute_patch_stats=compute_patch_and_phys,
        device=device,
        batch_size=batch_size,
        num_workers=nw,
    )
    emb = {"color_pre": c_pre, "color_post": c_post}
    if compute_patch_and_phys and c_chg is not None:
        emb["changed"] = c_chg

    if include_gray:
        g_pre, g_post, _, _ = run_pairwise_extraction(
            backbone, pairs_df, b_info, mode="gray3", compute_patch_stats=False, device=device, batch_size=batch_size, num_workers=nw
        )
        emb["gray_pre"] = g_pre
        emb["gray_post"] = g_post

    art_df = None
    phys_df = None
    if compute_patch_and_phys:
        art_rows = []
        phys_rows = []
        for _, row in tqdm(pairs_df.iterrows(), total=len(pairs_df), desc="Artifak & phys_v4"):
            rgb_p = cv2.cvtColor(cv2.imread(row["pre_path"]), cv2.COLOR_BGR2RGB)
            rgb_q = cv2.cvtColor(cv2.imread(row["post_path"]), cv2.COLOR_BGR2RGB)
            mp = compute_artifact_and_phys(rgb_p)
            mq = compute_artifact_and_phys(rgb_q)
            r = {"Id": row["Id"], **{f"{k}_pre": v for k, v in mp.items()}, **{f"{k}_post": v for k, v in mq.items()}}
            unrel = (mp["is_gray"] > 0.5) or (mq["is_gray"] > 0.5) or (mp["tint_score"] > TINT_NORM_THR) or (mq["tint_score"] > TINT_NORM_THR)
            r["color_unreliable_flag"] = float(unrel)
            r["delta_exg"] = mq["exg"] - mp["exg"]
            r["delta_water"] = mq["water"] - mp["water"]
            r["ash_gain"] = mq["ash_ratio"] - mp["ash_ratio"]
            r["sat_drop"] = mp["sat"] - mq["sat"]
            r["dark_char_gain"] = mq["dark_char"] - mp["dark_char"]
            r["delta_edge"] = mq["edge"] - mp["edge"]
            r["edge_ratio"] = (mq["edge"] + 1e-4) / (mp["edge"] + 1e-4)
            for ccol in ("delta_exg", "delta_water", "ash_gain", "sat_drop", "dark_char_gain"):
                r[f"{ccol}_nan"] = np.nan if unrel else r[ccol]
                r[f"{ccol}_imp0"] = 0.0 if unrel else r[ccol]
            art_rows.append(r)
            phys_rows.append(compute_phys_v4_pair(row["Id"], rgb_p, rgb_q))
        art_df = pd.DataFrame(art_rows)
        phys_full = pd.DataFrame(phys_rows)
        phys_df = phys_full[[c for c in phys_full.columns if c != "_raw_d_bright"]]

    return {
        "emb": emb,
        "art": art_df,
        "patch": patch_df,
        "phys": phys_df,
        "meta": b_info,
    }


def extract_embeddings_for_seno(
    pairs_df: pd.DataFrame,
    weights_dir: str = None,
    backbone_choice: str = BACKBONE_CHOICE,
    include_gray: bool = False,
    batch_size: int = BATCH_SIZE,
):
    """Fungsi terkunci untuk 01_model.ipynb & 02_submission.ipynb (offline load dari weights/)."""
    set_reproducible_seed(SEED)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    backbone, b_info = load_dino_backbone(
        backbone_choice=backbone_choice,
        device=device,
        local_weights_dir=weights_dir,
        save_weights_dir=None,
    )
    if ALLOW_DATAPARALLEL and device == "cuda" and torch.cuda.device_count() > 1:
        backbone = nn.DataParallel(backbone)

    c_pre, c_post, _, _ = run_pairwise_extraction(
        backbone, pairs_df, b_info, mode="color_norm", compute_patch_stats=False, device=device, batch_size=batch_size
    )
    out = {
        "Id": pairs_df["Id"].astype(str).tolist(),
        "dino_color_pre": c_pre,
        "dino_color_post": c_post,
        "meta": b_info,
    }
    if include_gray:
        g_pre, g_post, _, _ = run_pairwise_extraction(
            backbone, pairs_df, b_info, mode="gray3", compute_patch_stats=False, device=device, batch_size=batch_size
        )
        out["dino_gray_pre"] = g_pre
        out["dino_gray_post"] = g_post
    return out


In [ ]:
sub = pd.read_csv(SAMPLE_SUB)
ids = sub["Id"].to_numpy()
N = len(ids)
assert sub["Id"].is_unique and sub.columns[1] == "Phenomenon"

if MODE_UJI:
    print("PERINGATAN MODE_UJI: embedding dimuat dari file tersimpan, BUKAN diekstrak dari citra")
    emb = {b: np.load(FITUR_TERSIMPAN / f"dino_{b}.npy") for b in sorted(set(FITUR_BLOK) | {"color_pre", "color_post"})}
else:
    assert "ekstrak_semua" in globals(), "fungsi ekstrak_semua dari Hafizh belum ditempel di sel atas"
    emb = ekstrak_semua(DATA_DIR, ids, weights_dir=WEIGHTS_DIR, compute_patch_and_phys=False)["emb"]

for b in sorted(set(FITUR_BLOK) | {"color_pre", "color_post"}):
    assert emb[b].shape == (N, 1024) and not np.isnan(emb[b]).any(), f"embedding {b} bermasalah"
print("N =", N, "| blok embedding:", sorted(emb))


## 2. [B — Fahmi] Cluster & nama kelas (tanpa fit ulang)
Clustering dijalankan Fahmi di `fahmi/B_kejadian.ipynb` (Spectral K=40 di ruang Z, lalu cluster campur 2, 10, 29, 39 dipecah
menjadi sub-cluster 40–49). Di sini hasilnya **dimuat** dan dicek integritasnya lewat hash, tidak di-fit ulang.

Nama kelas per cluster ditulis eksplisit di kamus berikut (salinan persis `fahmi/v3/dictionary_nama_cluster.py`).
Alasan visual tiap cluster ada di `cluster_names.json` (kolom `alasan`).


In [ ]:
# Kamus nama cluster v3-7 (dibuat dari cluster_names.json). Salin ke notebook final/01_model.ipynb (FAQ no. 7: kamus eksplisit).
# Nama kelas ditentukan dari montase tiap cluster (penilai: AI Claude, tanpa tinjauan tim). Bukan dari leaderboard.
# ID 0-39 = cluster Spectral K=40 (kecuali induk campur 2, 10, 29, 39 yang dipecah); ID 40-49 = sub-cluster hasil pemecahan.
NAMA_CLUSTER = {
    0: "Volcano",
    1: "Hurricane",
    3: "Volcano",
    4: "Tsunami",
    5: "Earthquake",
    6: "Volcano",
    7: "Hurricane",
    8: "Tsunami",
    9: "Fire",
    11: "Tsunami",
    12: "Fire",
    13: "Fire",
    14: "Earthquake",
    15: "Fire",
    16: "Hurricane",
    17: "Fire",
    18: "Hurricane",
    19: "Fire",
    20: "Fire",
    21: "Hurricane",
    22: "Fire",
    23: "Hurricane",
    24: "Hurricane",
    25: "Earthquake",
    26: "Fire",
    27: "Earthquake",
    28: "Fire",
    30: "Earthquake",
    31: "Earthquake",
    32: "Fire",
    33: "Earthquake",
    34: "Hurricane",
    35: "Hurricane",
    36: "Fire",
    37: "Hurricane",
    38: "Hurricane",
    40: "Hurricane",  # sub 2a
    41: "Tsunami",  # sub 2b
    42: "Hurricane",  # sub 10a
    43: "Volcano",  # sub 10b
    44: "Hurricane",  # sub 29a
    45: "Hurricane",  # sub 29b
    46: "Earthquake",  # sub 29c
    47: "Hurricane",  # sub 39a
    48: "Hurricane",  # sub 39b
    49: "Hurricane",  # sub 39c
}
YAKIN_CLUSTER = {
    0: "yakin",
    1: "yakin",
    3: "ragu",
    4: "yakin",
    5: "yakin",
    6: "ragu",
    7: "yakin",
    8: "ragu",
    9: "ragu",
    11: "ragu",
    12: "ragu",
    13: "ragu",
    14: "yakin",
    15: "yakin",
    16: "ragu",
    17: "ragu",
    18: "yakin",
    19: "yakin",
    20: "ragu",
    21: "yakin",
    22: "ragu",
    23: "yakin",
    24: "ragu",
    25: "ragu",
    26: "yakin",
    27: "ragu",
    28: "yakin",
    30: "ragu",
    31: "yakin",
    32: "ragu",
    33: "ragu",
    34: "yakin",
    35: "ragu",
    36: "yakin",
    37: "yakin",
    38: "yakin",
    40: "ragu",
    41: "yakin",
    42: "ragu",
    43: "ragu",
    44: "ragu",
    45: "ragu",
    46: "ragu",
    47: "ragu",
    48: "ragu",
    49: "ragu",
}

INDUK_DIPECAH = {2: [40, 41], 10: [42, 43], 29: [44, 45, 46], 39: [47, 48, 49]}
EVENT_ID_SHA1_40CLUSTER = "9517d553f3ad1ea535c7175b7a60f3c85bba2652"  # hash assignment 40-cluster asli (event_id induk, int32)


In [ ]:
events = pd.read_parquet(EVENTS_PATH)
assert len(events) == N and (events["Id"].to_numpy() == ids).all(), "urutan Id events.parquet beda dengan sample_submission"
event_id = events["event_id"].to_numpy()
assert set(event_id) == set(NAMA_CLUSTER) == set(YAKIN_CLUSTER), "ada cluster tanpa nama atau nama tanpa cluster"
assert set(NAMA_CLUSTER.values()) <= set(KELAS)

# Kamus = cluster_names.json = kolom di events.parquet
with open(CLUSTER_NAMES_PATH, encoding="utf-8") as f:
    cluster_names = {int(k): v for k, v in json.load(f).items()}
assert {k: v["kelas"] for k, v in cluster_names.items()} == NAMA_CLUSTER, "kamus beda dengan cluster_names.json"
assert {k: v["yakin"] for k, v in cluster_names.items()} == YAKIN_CLUSTER, "yakin beda dengan cluster_names.json"
kelas_cluster = pd.Series(event_id).map(NAMA_CLUSTER).to_numpy()
yakin_cluster = pd.Series(event_id).map(YAKIN_CLUSTER).to_numpy()
assert (kelas_cluster == events["kelas_cluster"].to_numpy()).all(), "kelas_cluster di events.parquet beda dengan kamus"
assert (yakin_cluster == events["yakin_cluster"].to_numpy()).all(), "yakin_cluster di events.parquet beda dengan kamus"

# Cek integritas: rekonstruksi event_id induk 40-cluster lalu hash (wajib int32)
ev = events.set_index("Id")
sub_ev = pd.read_parquet(SUB_EVENTS_PATH)
induk = ev["event_id"].copy()
induk.loc[sub_ev["Id"].values] = sub_ev["event_id_induk"].values
EVENT_ID_SHA1 = hashlib.sha1(induk.values.astype("int32").tobytes()).hexdigest()
assert EVENT_ID_SHA1 == EVENT_ID_SHA1_40CLUSTER, f"hash event_id beda: {EVENT_ID_SHA1}"
print("Hash event_id 40-cluster cocok:", EVENT_ID_SHA1)

kelas_hilang = [k for k in KELAS if k not in set(NAMA_CLUSTER.values())]
if kelas_hilang:
    print("PERINGATAN: kelas tanpa cluster ->", kelas_hilang, "(tidak akan pernah diprediksi)")
ringkas = pd.DataFrame({"kelas": kelas_cluster, "yakin": yakin_cluster, "event_id": event_id})
pd.DataFrame({
    "n_cluster": ringkas.groupby("kelas")["event_id"].nunique(),
    "n_cluster_ragu": ringkas[ringkas["yakin"] == "ragu"].groupby("kelas")["event_id"].nunique(),
    "n_foto": ringkas["kelas"].value_counts(),
}).reindex(KELAS).fillna(0).astype(int)


## 3. [C — Seno] Seed → label spreading → classifier
Sama dengan `seno/C_model_run5.ipynb` (run5_final). Ruang Z dibangun seperti B dengan `pca.joblib` + `mean_x.npy` milik B.
Seed = foto dengan `kedekatan` ≥ 0,5. Bobot sampel = kedekatan × (1,0 yakin / 0,5 ragu).
Classifier hanya memakai embedding per foto (tanpa `event_id`).


In [ ]:
def l2(a):
    a = a.astype(np.float64)
    return a / np.linalg.norm(a, axis=1, keepdims=True)

def bangun_fitur(emb, blok):
    # fitur per foto: L2 per blok embedding, digabung (tanpa event_id)
    return np.hstack([l2(emb[b]) for b in blok])

# Ruang Z seperti B: L2 per blok -> gabung pre+post -> kurangi mean_x B -> PCA B -> bulatkan
X_b = np.hstack([l2(emb["color_pre"]), l2(emb["color_post"])])
pca_b = joblib.load(PCA_B_PATH)
mean_x = np.load(MEAN_X_PATH)
assert np.allclose(mean_x, X_b.mean(axis=0), atol=1e-6), "mean_x B beda dengan embedding: cek ekstraksi"
Z = np.round(pca_b.transform(X_b - mean_x), B_ROUND)

kedekatan = events["kedekatan"].to_numpy(dtype=float)
assert np.isfinite(kedekatan).all() and kedekatan.min() >= 0 and kedekatan.max() <= 1
is_seed = kedekatan >= SEED_KEDEKATAN_MIN
bobot = kedekatan * pd.Series(yakin_cluster).map(BOBOT_YAKIN).to_numpy(dtype=float)

y = np.full(N, -1)
y[is_seed] = pd.Categorical(kelas_cluster[is_seed], categories=KELAS).codes
ls = LabelSpreading(kernel="knn", n_neighbors=LS_KNN, alpha=LS_ALPHA, max_iter=1000).fit(Z, y)
y_latih = np.array(KELAS)[ls.classes_][ls.label_distributions_.argmax(axis=1)]
print(f"Seed: {int(is_seed.sum())} foto | kelas terisi {len(set(y_latih))}/6")

X_c = bangun_fitur(emb, FITUR_BLOK)
model = Pipeline([
    ("scaler", StandardScaler()),
    ("pca", PCA(C_PCA, random_state=SEED)),
    ("clf", LogisticRegression(class_weight="balanced", C=LR_C, max_iter=5000)),
]).fit(X_c, y_latih, clf__sample_weight=bobot)

pred = model.predict(X_c)
proba = model.predict_proba(X_c)
pd.Series(pred).value_counts().reindex(KELAS).fillna(0).astype(int).to_frame("n_prediksi")


## 4. Simpan model

In [ ]:
MODEL_DIR.mkdir(exist_ok=True)
VERSI_LIB = {"python": platform.python_version(), "numpy": np.__version__,
             "pandas": pd.__version__, "scikit-learn": sklearn.__version__, "joblib": joblib.__version__}

joblib.dump({"pipeline": model, "fitur_blok": FITUR_BLOK, "l2_per_blok": True,
             "kelas": [str(k) for k in model.classes_], "versi_lib": VERSI_LIB},
            MODEL_DIR / "model.joblib")

pred_train = pd.DataFrame({"Id": ids, "Phenomenon": pred, "label_latih": y_latih, "is_seed": is_seed,
                           "event_id": event_id, "bobot": bobot.round(6)})
for i, k in enumerate(model.classes_):
    pred_train[f"proba_{k}"] = proba[:, i].round(6)
pred_train.to_csv(MODEL_DIR / "pred_train.csv", index=False)

meta = {
    "tanggal": datetime.now().strftime("%Y-%m-%d %H:%M"),
    "mode_uji": MODE_UJI,
    "b": {"sumber": "fahmi/v3 (v3-7)", "n_cluster": len(NAMA_CLUSTER), "event_id_sha1_40cluster": EVENT_ID_SHA1,
          "round": B_ROUND, "penamaan": "per cluster dari montase oleh AI (Claude), tanpa tinjauan tim; kamus NAMA_CLUSTER"},
    "c": {"seed_kedekatan_min": SEED_KEDEKATAN_MIN, "bobot_yakin": BOBOT_YAKIN, "ls_knn": LS_KNN, "ls_alpha": LS_ALPHA,
          "fitur_blok": FITUR_BLOK, "c_pca": C_PCA, "lr_C": LR_C, "class_weight": "balanced"},
    "seed": SEED,
    "distribusi_prediksi": pd.Series(pred).value_counts().reindex(KELAS).fillna(0).astype(int).to_dict(),
    "versi_lib": VERSI_LIB,
}
with open(MODEL_DIR / "meta.json", "w", encoding="utf-8") as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)
print("Disimpan ke", MODEL_DIR, ":", sorted(os.listdir(MODEL_DIR)))

In [ ]:
# Cek: prediksi sama dengan run C (jika foldernya ada)
if C_ACUAN.exists():
    ref = pd.read_csv(C_ACUAN / "pred_train.csv")
    assert (ref["Id"].to_numpy() == ids).all()
    sama = (ref["Phenomenon"].to_numpy() == pred).mean()
    print(f"Sama dengan {C_ACUAN}: {sama:.4f}")
    assert sama == 1.0, "prediksi beda dengan run C"

# requirements.txt dari lingkungan yang menjalankan notebook ini (hanya versi final)
PAKET = ["numpy", "pandas", "pyarrow", "scipy", "scikit-learn", "joblib",
         "torch", "torchvision", "transformers", "timm", "opencv-python", "scikit-image", "Pillow"]
baris = []
for p in PAKET:
    try:
        baris.append(f"{p}=={version(p)}")
    except PackageNotFoundError:
        pass
if MODE_UJI:
    print("MODE_UJI: requirements.txt TIDAK ditulis. Versi saat ini:", baris)
else:
    Path("requirements.txt").write_text("\n".join(baris) + "\n", encoding="utf-8")
    print("requirements.txt ditulis:", baris)